[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/ruppeiner/ruppeiner.ipynb)

# The Ruppeiner Metric Near the Critical Point

This example computes the fluctuation metric of a van der Waals fluid, the cost of a small variation of its temperature and volume away from a stable state: the larger the cost, the less likely the fluid fluctuates that far. Near the critical point the fluid's pressure hardly changes with its volume, so variations of the volume cost almost nothing and the fluid fluctuates wildly, which shows as the ellipses of equal cost stretching along the volume and as the metric's curvature growing without bound.

A state is modelled as an antivector of the plane spanned by temperature, `mv.t.dual()`, and volume, `mv.v.dual()`. The metric is a map from a variation of the state to a gradient, `Gradient <- State`, which pairs with the variation to its cost, `metric(state)(variation) & variation`: it runs the other way to a mobility, which maps a gradient to a change of state. It is built from the Helmholtz energy's second derivative in its own variables, temperature and volume, the thermal part with its sign turned and both over the temperature; in these variables it has no cross term. The ellipses of equal cost are the level sets of that pairing, drawn as conics.

In the line-element notation of Riemannian geometry, with the partial derivatives of thermodynamics whose subscript names the quantity held fixed, the metric reads $ds^2 = \frac{C_V}{T^2}\,dT^2 - \frac{1}{T}\left(\frac{\partial p}{\partial V}\right)_T dV^2$, and the probability of a fluctuation falls as $e^{-ds^2/2}$. Its scalar curvature, here $R = 2K$, positive on a sphere (Ruppeiner's sign convention is the opposite), grows as $(T - T_c)^{-2}$ toward the critical point along the critical volume.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.ruppeiner import render

ga = Algebra("t0v0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                                   # Scalar+
State = ga.gatype.antivector()                                # Antivector-
Gradient = ga.gatype.vector()                                 # Vector+
Metric = ga.gatype((Gradient, State))                         # Gradient+ <- State-

# The two readings a state is described by; temperature and volume are in their critical values, pressure in
# R T_c / V_c and energy in R T_c, so the critical pressure is 3/8.
temperature = mv.t & State                                    # [] Scalar+ <- State-
volume = mv.v & State                                         # [] Scalar+ <- State-
# The fluid: the strength of its molecules' attraction, the volume they exclude, and its heat capacity at fixed volume.
attraction, exclusion, capacity = 9 / 8, 1 / 3, 1.5

## The fluctuation metric

The Helmholtz energy's second derivative is a form on pairs of variations of the state. On two variations of temperature it is minus the heat capacity over the temperature, and on two variations of volume minus the pressure's slope along the volume. The fluctuation metric takes these as the costs of the two variations, the thermal one with its sign turned and both over the temperature, as a map from a variation to the gradient that reads its cost. At a state well above the critical point both costs are positive.

In [ ]:
@jax.jit
def pressure(state: State) -> Scalar:
    """The van der Waals equation of state."""
    return temperature(state) / (volume(state) - exclusion) - attraction / volume(state).squared()

def helmholtz_energy(state: State) -> Scalar:
    t, v = temperature(state), volume(state)                                 # [...] Scalar+, Scalar+
    return -capacity * t * t.log() - t * (v - exclusion).log() - attraction / v

@jax.jit
def metric(state: State) -> Metric:
    """The cost of variations about a state, as a map from a variation to a gradient."""
    hessian = derivative(derivative(helmholtz_energy))(state)                 # [...] Scalar+ <- (State-, State-)
    # The thermal part with its sign turned; in temperature and volume there is no cross term.
    thermal = -hessian(mv.t.dual(), mv.t.dual()) / temperature(state)        # [...] Scalar+
    mechanical = hessian(mv.v.dual(), mv.v.dual()) / temperature(state)      # [...] Scalar+
    return mv.t * thermal * temperature + mv.v * mechanical * volume

warm = mv.t.dual() * 1.8 + mv.v.dual()                                       # [] State-

In [ ]:
print("cost of a unit variation of temperature:", (metric(warm)(mv.t.dual()) & mv.t.dual()).to_array())
print("cost of a unit variation of volume:     ", (metric(warm)(mv.v.dual()) & mv.v.dual()).to_array())

## The metric's curvature

The scalar curvature measures how far the cost of fluctuations departs from that of a flat plane; without the molecules' attraction and excluded volume, for the ideal gas, it vanishes. For a metric without a cross term whose thermal entry depends on the temperature alone, it follows from the mechanical entry's changes along the temperature and the change of the squared area the two entries span. The chart shows the plane of temperature and volume: shaded below the black spinodal, where the mechanical entry turns negative and the uniform fluid is unstable; above it, teal ellipses of equal cost about a lattice of states, and the grey-blue level lines of the curvature, crowding toward the critical point at the spinodal's peak. The state above is orange.

In [ ]:
@jax.jit
def curvature(state: State) -> Scalar:
    """The metric's scalar curvature, twice its Gaussian curvature, positive on a sphere."""
    thermal = lambda point: metric(point)(mv.t.dual()) & mv.t.dual()
    mechanical = lambda point: metric(point)(mv.v.dual()) & mv.v.dual()
    area_squared = lambda point: thermal(point) * mechanical(point)
    slope = derivative(mechanical)(state)(mv.t.dual())                                    # [...] Scalar+
    slope_change = derivative(derivative(mechanical))(state)(mv.t.dual(), mv.t.dual())   # [...] Scalar+
    area_change = derivative(area_squared)(state)(mv.t.dual())                           # [...] Scalar+
    return -slope_change / area_squared(state) + slope * area_change / (2 * area_squared(state).squared())

render.draw_landscape(metric, curvature, pressure, warm);

## Toward the critical point

The state cools along the critical volume from 1.8 to 1.02 times the critical temperature. Its ellipse of equal cost widens along the volume while hardly changing along the temperature, and on the right its isotherm's slope at the state, the mechanical entry of the metric times minus the temperature, flattens.

In [ ]:
n_frames, frame_duration_ms = 48, 80
temperatures = np.linspace(1.8, 1.02, n_frames)

In [ ]:
def approach(temperatures: np.ndarray) -> Iterator[State]:
    """States at the critical volume cooling through the temperatures toward the critical one."""
    for frame in range(len(temperatures)):
        yield mv.t.dual() * temperatures[frame] + mv.v.dual()

show_animation(render.fluctuations(metric, curvature, pressure, approach(temperatures)), "ruppeiner_fluctuations",
               duration_ms=frame_duration_ms)

Along the critical volume the curvature grows as the inverse square of the distance above the critical temperature: on logarithmic scales its line runs parallel to the dashed one of slope $-2$.

In [ ]:
# Distances above the critical temperature, along the critical volume.
offsets = np.geomspace(0.002, 0.8, 100)
above_critical = mv.t.dual() * (1 + offsets) + mv.v.dual()                   # [samples] State-
render.scaling(above_critical, curvature);

In [ ]:
# checks
temperatures = np.array([1.02, 1.2, 1.8])
states = mv.t.dual() * temperatures + mv.v.dual() * np.array([1, 0.8, 1.5])
# The thermal entry is the heat capacity over the squared temperature, and there is no cross term.
thermal = metric(states)(mv.t.dual()) & mv.t.dual()
np.testing.assert_allclose(thermal.to_array(), capacity / temperatures ** 2, atol=1e-10)
np.testing.assert_allclose((metric(states)(mv.t.dual()) & mv.v.dual()).to_array(), 0, atol=1e-10)
# Close to the critical point the curvature is a third of the inverse square of the distance above it.
near = curvature(mv.t.dual() * 1.002 + mv.v.dual())
assert abs(float(near.to_array()) * 3 * 0.002 ** 2 - 1) < 0.01